In [7]:
!pip install gradio transformers torch pillow

In [8]:

import gradio as gr
from transformers import pipeline

# Load AI animal recognition model
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

# Animal information database
animal_info = {
    "dog": {
        "category": "Mammal",
        "habitat": "Homes, forests and grasslands",
        "food": "Meat and animal-based food",
        "description": "Dogs are friendly domestic animals known for loyalty."
    },
    "cat": {
        "category": "Mammal",
        "habitat": "Homes and various environments",
        "food": "Meat and fish",
        "description": "Cats are small, furry animals known for agility."
    },
    "bird": {
        "category": "Bird",
        "habitat": "Forests, trees and wetlands",
        "food": "Seeds, fruits and insects",
        "description": "Birds have feathers, wings and beaks."
    },
    "lion": {
        "category": "Mammal",
        "habitat": "African grasslands and savannas",
        "food": "Meat",
        "description": "Lions are large wild cats known as the king of the jungle."
    },
    "tiger": {
        "category": "Mammal",
        "habitat": "Forests and grasslands",
        "food": "Meat",
        "description": "Tigers are powerful wild cats with black stripes."
    },
    "elephant": {
        "category": "Mammal",
        "habitat": "Forests and grasslands",
        "food": "Grass, leaves, fruits and bark",
        "description": "Elephants are large animals with trunks and tusks."
    },
    "horse": {
        "category": "Mammal",
        "habitat": "Grasslands and farms",
        "food": "Grass and hay",
        "description": "Horses are strong animals commonly used for riding."
    },
    "cow": {
        "category": "Mammal",
        "habitat": "Farms and grasslands",
        "food": "Grass and hay",
        "description": "Cows are domestic farm animals that produce milk."
    },
    "bear": {
        "category": "Mammal",
        "habitat": "Forests and mountains",
        "food": "Plants, fruits, insects and meat",
        "description": "Bears are large mammals with strong bodies."
    },
    "rabbit": {
        "category": "Mammal",
        "habitat": "Grasslands and forests",
        "food": "Grass, vegetables and leaves",
        "description": "Rabbits are small furry animals with long ears."
    },
    "snake": {
        "category": "Reptile",
        "habitat": "Forests, deserts and grasslands",
        "food": "Small animals and rodents",
        "description": "Snakes are reptiles that move without legs."
    },
    "butterfly": {
        "category": "Insect",
        "habitat": "Gardens and forests",
        "food": "Flower nectar",
        "description": "Butterflies are colorful insects with beautiful wings."
    },
    "frog": {
        "category": "Amphibian",
        "habitat": "Ponds and wetlands",
        "food": "Insects and small organisms",
        "description": "Frogs are amphibians that can live on land and in water."
    },
    "monkey": {
        "category": "Mammal",
        "habitat": "Forests and trees",
        "food": "Fruits, leaves and insects",
        "description": "Monkeys are intelligent animals that climb trees."
    },
    "fish": {
        "category": "Aquatic animal",
        "habitat": "Rivers, lakes and oceans",
        "food": "Algae, plankton and smaller aquatic organisms",
        "description": "Fish are aquatic animals that breathe through gills."
    }
}

# Match model labels with animal names
animal_aliases = {
    "dog": ["dog", "retriever", "terrier", "poodle", "spaniel", "shepherd", "chihuahua", "beagle", "husky"],
    "cat": ["cat", "tabby", "siamese", "persian"],
    "bird": ["bird", "parrot", "eagle", "sparrow", "owl", "finch", "peacock", "macaw"],
    "lion": ["lion"],
    "tiger": ["tiger"],
    "elephant": ["elephant"],
    "horse": ["horse", "pony", "zebra"],
    "cow": ["cow", "ox"],
    "bear": ["bear", "panda"],
    "rabbit": ["rabbit", "hare"],
    "snake": ["snake", "cobra", "python", "viper"],
    "butterfly": ["butterfly", "moth"],
    "frog": ["frog", "toad"],
    "monkey": ["monkey", "baboon", "macaque", "chimpanzee"],
    "fish": ["fish", "goldfish", "shark", "ray"]
}

def recognize_animal(image):

    if image is None:
        return "Please upload an animal image first."

    try:
        image = image.convert("RGB")

        predictions = classifier(image, top_k=5)

        result = None
        animal_name = None

        for prediction in predictions:
            label = prediction["label"].replace("_", " ").lower()

            for animal, keywords in animal_aliases.items():
                if any(word in label for word in keywords):
                    result = prediction
                    animal_name = animal
                    break

            if result:
                break

        if result is None:
            top = predictions[0]
            return (
                "Animal Not Confirmed\n\n"
                f"AI Prediction: {top['label']}\n"
                f"Confidence: {top['score'] * 100:.2f}%\n\n"
                "Please upload a clearer animal image."
            )

        info = animal_info[animal_name]

        return f"""
AI ANIMAL RECOGNITION RESULT

Animal Name: {animal_name.title()}

Category: {info['category']}

Habitat: {info['habitat']}

Food Habits: {info['food']}

Description: {info['description']}

AI Confidence: {result['score'] * 100:.2f}%

TOP 5 AI PREDICTIONS:

""" + "\n".join(
            f"{i+1}. {p['label']} - {p['score']*100:.2f}%"
            for i, p in enumerate(predictions)
        ) + "\n\nNote: This is an AI prediction. Please verify the result."

    except Exception as e:
        return f"Error processing image: {str(e)}"


with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🐾 AI Animal Recognition Chatbot

    Upload an animal image and let AI identify it!
    """)

    with gr.Row():

        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Your Animal Image"
            )

            analyze_button = gr.Button(
                "🐾 Recognize Animal",
                variant="primary"
            )

        with gr.Column():
            result_output = gr.Textbox(
                label="Chatbot Response",
                lines=15,
                interactive=False
            )

    analyze_button.click(
        fn=recognize_animal,
        inputs=image_input,
        outputs=result_output
    )

app.launch(share=True)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_2477/1991528104.py:185: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0face344caa98c122.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
